In [2]:
import base64
import requests
from pathlib import Path
from PIL import Image
import io

# ----------------------------
# CONFIG
# ----------------------------

ACCOUNT_HASH = "LJW9TBw5H2BwGc8uUBxFZA"
image_ids = [
]

with open('cf_image_ids.txt', 'r') as r:
    image_ids = [i.strip() for i in r.readlines()]

OUTPUT_HTML = "images.html"

# ----------------------------
# HELPERS
# ----------------------------

def image_url(image_id: str) -> str:
    return f"https://imagedelivery.net/{ACCOUNT_HASH}/{image_id}/thumbxsm"


def fetch_image_as_base64(url: str) -> str:
    response = requests.get(url, timeout=15)
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "image/jpeg")

    img = Image.open(io.BytesIO(response.content))

    buf = io.BytesIO()
    img.save(buf, format="WEBP", quality=60)
    webp_bytes = buf.getvalue()

    # encoded = base64.b64encode(response.content).decode("utf-8")
    encoded = base64.b64encode(webp_bytes).decode("utf-8")
    content_type = "image/webp"
    
    return f"data:{content_type};base64,{encoded}"




In [3]:
html_imgs = []
img_tags = []


In [4]:

# ----------------------------
# MAIN
# ----------------------------
fails = []
for i, image_id in enumerate(image_ids):
    if i <= len(img_tags): continue
    print(f'{i}/{len(image_ids)}', end='\r')
    url = image_url(image_id)

    try:
        data_uri = fetch_image_as_base64(url)
    except:
        fails.append([i, image_id])
        
    img_tags.append(
        f'<img src="{data_uri}" loading="lazy" id="img__{image_id}"/>'
    )

html_content = '''
<div id="images">
    '''+"".join(img_tags)+'''
</div>
<script>
document.querySelectorAll('img[id^=img__]').forEach(
    img=>{
        img.onclick=function(img_el){
            let img_id = img_el.target.id.replace('img__', '');
             window.open(`https://imagedelivery.net/LJW9TBw5H2BwGc8uUBxFZA/${img_id}/public`, '_blank');

        }
    }
)
</script>
'''

Path(OUTPUT_HTML).write_text(html_content, encoding="utf-8")

print(f"Saved {OUTPUT_HTML}")

Saved images.html


In [1]:
len(img_tags)

NameError: name 'img_tags' is not defined